# Trees

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Represent a tree with a parent array or with nodes, and find its levels
- Build a binary search tree and traverse it in-order, pre-order and post-order
- Explain why a BST's height decides its speed, and when it turns into a list
- Answer prefix questions with a trie, or with sorted words
- Add values up a tree with a post-order traversal

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## In short

A **tree** is a hierarchy: one **root**, and every other node has exactly one **parent**. Org charts, folders, web pages and the expressions a compiler reads are trees. A node's **depth** is its distance from the root, and the tree's **height** is the depth of its deepest node. A tree can live in a **parent array** (`parent[i]`) or in **nodes** that point to their children.

**A tree in a parent array:** each node's depth comes from its parent's, and its children are counted in one pass.

In [ ]:
int[] parent = {-1, 0, 0, 1, 1, 2, 4, 4}; // node 0 is the root
int[] depth = new int[parent.length];
int[] children = new int[parent.length];
for (int i = 1; i < parent.length; i++) {
    depth[i] = depth[parent[i]] + 1;
    children[parent[i]]++;
}
for (int i = 0; i < parent.length; i++) {
    IO.println("node " + i + ": depth " + depth[i] + ", children " + children[i] + (children[i] == 0 ? " (a leaf)" : ""));
}

### ✏️ Exercise: Depth from a parent array

parent[i] is the parent of node i, and the root has parent -1. Walk up from a node to the root and count the steps.

In [ ]:
int depth(int[] parent, int node) {
    // TODO: follow parent[] up to the root, counting steps
    return 0;
}

void main() {
    check("the root", depth(new int[] {-1, 0, 0, 1}, 0), 0);
    check("a child", depth(new int[] {-1, 0, 0, 1}, 2), 1);
    check("a grandchild", depth(new int[] {-1, 0, 0, 1}, 3), 2);
}

<details>
<summary>Show an answer</summary>

```java
int depth(int[] parent, int node) {
    int steps = 0;
    while (parent[node] != -1) {
        node = parent[node];
        steps++;
    }
    return steps;
}

void main() {
    check("the root", depth(new int[] {-1, 0, 0, 1}, 0), 0);
    check("a child", depth(new int[] {-1, 0, 0, 1}, 2), 1);
    check("a grandchild", depth(new int[] {-1, 0, 0, 1}, 3), 2);
}
```

</details>

A **binary search tree** (BST) keeps smaller keys on the left of every node and bigger keys on the right, so a search walks down a single path: O(height).

![A binary search tree and its in-order traversal](bst.svg)

#### 🎞️ Building and searching a binary search tree

![Inserting 8, 3, 10 and 6 into a binary search tree, then searching 6 along the path 8, 3, 6](bst-steps.svg)

Random insertions give a height of about 2 log₂ n. Sorted insertions build a chain of height n: a slow linked list. That's why Java's `TreeMap` rebalances itself to stay O(log n).

**Random against sorted insertions.** This cell only tracks how deep each new key lands; BST Builder asks for the whole tree.

In [ ]:
int n = 2000;
for (String order : new String[] {"random", "sorted"}) {
    int[] keys = order.equals("random") ? new Random(5).ints(n, 0, 1_000_000).distinct().toArray() : java.util.stream.IntStream.range(0, n).toArray();
    TreeMap<Integer, Integer> depthOf = new TreeMap<>(); // key -> its depth in the BST
    int height = 0;
    for (int key : keys) {
        // In a BST a new key hangs below its closest neighbours: the deeper of the two.
        Map.Entry<Integer, Integer> below = depthOf.lowerEntry(key);
        Map.Entry<Integer, Integer> above = depthOf.higherEntry(key);
        int depth = 1 + Math.max(below == null ? 0 : below.getValue(), above == null ? 0 : above.getValue());
        depthOf.put(key, depth);
        height = Math.max(height, depth);
    }
    IO.println(String.format("%s insertions of %d keys: height %d (log2 n is about %.0f)", order, keys.length, height, Math.log(keys.length) / Math.log(2)));
}

Three **traversals** visit every node:

- **pre-order** (node, left, right) copies a tree;
- **in-order** (left, node, right) gives a BST's keys sorted;
- **post-order** (left, right, node) finishes the children first, which is how a folder adds up its contents.

### ✏️ Exercise: Add up a tree

A node holds a value and two children (null when missing). The sum of a tree is its value plus the sums of its two subtrees: a short recursion.

In [ ]:
record Node(int value, Node left, Node right) {}

int sum(Node n) {
    // TODO: return 0 for null, otherwise the value plus both subtrees
    return 0;
}

void main() {
    check("one node", sum(new Node(5, null, null)), 5);
    check("three nodes", sum(new Node(8, new Node(3, null, null), new Node(10, null, null))), 21);
    check("empty", sum(null), 0);
}

<details>
<summary>Show an answer</summary>

```java
record Node(int value, Node left, Node right) {}

int sum(Node n) {
    if (n == null) {
        return 0;
    }
    return n.value() + sum(n.left()) + sum(n.right());
}

void main() {
    check("one node", sum(new Node(5, null, null)), 5);
    check("three nodes", sum(new Node(8, new Node(3, null, null), new Node(10, null, null))), 21);
    check("empty", sum(null), 0);
}
```

</details>

A **trie** is a tree of letters: each path from the root spells a prefix, so counting the words that start with "car" takes three steps down.

**Words by prefix in a sorted set:** the words starting with "ca" sit between "ca" and "ca{" (`{` comes right after `z`).

In [ ]:
TreeSet<String> words = new TreeSet<>(List.of("car", "card", "care", "cart", "cat", "dog", "do", "dot"));
for (String prefix : new String[] {"ca", "car", "do", "x"}) {
    IO.println(prefix + " -> " + words.subSet(prefix, prefix + "{"));
}

Watch out: recursion on a deep, chain-shaped tree can overflow the call stack.

**Deep recursion:** walking a chain of 100,000 nodes recursively overflows the call stack; a loop doesn't.

In [ ]:
// shared: a chain where node i's parent is i - 1
int[] chainParent(int n) {
    int[] parent = new int[n];
    for (int i = 0; i < n; i++) {
        parent[i] = i - 1;
    }
    return parent;
}

int depthRecursive(int[] parent, int node) {
    return parent[node] < 0 ? 0 : 1 + depthRecursive(parent, parent[node]);
}

In [ ]:
int[] parent = chainParent(100_000);
try {
    IO.println("recursive: " + depthRecursive(parent, 99_999));
} catch (StackOverflowError e) {
    IO.println("recursive: StackOverflowError");
}
int depth = 0;
for (int node = 99_999; parent[node] >= 0; node = parent[node]) {
    depth++;
}
IO.println("loop: " + depth);

## Readings

- [Binary Search Tree](https://visualgo.net/en/bst) (VisuAlgo (National University of Singapore), 20 min): Insert your own keys, then try Search and the in-order traversal. Insert 1, 2, 3, 4, 5 into an empty tree to see it become a chain, and note what the AVL tree in the same module does to avoid that.
- [Introduction to Tree Algorithms](https://usaco.guide/silver/intro-tree) (USACO Guide, 12 min): The vocabulary (root, parent, subtree, depth), the three traversal orders, and the Subordinates example: a node's subtree size is one plus its children's, the same idea as a folder's total size. Pick Java in the site's settings to see Java code.

## Check yourself

1. Which traversal prints a BST's keys in order?
2. What do you get by inserting 1, 2, 3, 4, 5 into an empty BST, and what is its height?
3. Why does a folder's size need a post-order traversal?

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Org Chart** · Easy
- **BST Builder** · Medium
- **Autocomplete** · Medium
- **Folder Sizes** · Hard